# Comparative Analysis of GEO, AI Search Visibility, and RAG Benchmarks

This notebook converts the 23-paper literature table into a comparative analysis workbook.

**Important scoring rule used here:** the notebook ignores the original `Priority Label` column and classifies papers only by the numerical `Priority Score` using the user-defined criteria:

| Score | Meaning |
|---:|---|
| 9–10 | Must-read / central paper |
| 8–8.9 | Highly relevant, should include in memo |
| 7–7.9 | Useful supporting paper |
| 6–6.9 | Background only |
| <6 | Probably mention briefly or exclude |

The final output is an Excel workbook with benchmark comparison, metric crosswalk, similarity features, paper-pair similarity matrix, limitation heatmap, and finance/econ research gaps.

In [5]:
from pathlib import Path
import re
import math
import numpy as np
import pandas as pd

# Standard .xlsx export/styling dependency.
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.formatting.rule import ColorScaleRule
from openpyxl.utils import get_column_letter

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

## 1. Load the literature table

The code looks first in the current working directory, then in `/mnt/data/`, so the notebook can run both locally and in the ChatGPT sandbox.

In [6]:
from pathlib import Path
import pandas as pd
from IPython.display import display

INPUT_CSV_NAME = "GEO_AI_Search_Literature_Review_Table_FINAL_metrics_updated.csv"
INPUT_XLSX_NAME = "JunseoPark_GEO_AI_Search_Literature_Review.xlsx"

cwd = Path.cwd()

candidate_csv_paths = [
    cwd / "00_Memo" / INPUT_CSV_NAME,
    cwd.parent / "00_Memo" / INPUT_CSV_NAME,
    cwd / INPUT_CSV_NAME,
    Path("/mnt/data") / "00_Memo" / INPUT_CSV_NAME,
    Path("/mnt/data") / INPUT_CSV_NAME,
]

candidate_xlsx_paths = [
    cwd / "00_Memo" / INPUT_XLSX_NAME,
    cwd.parent / "00_Memo" / INPUT_XLSX_NAME,
    cwd / INPUT_XLSX_NAME,
    Path("/mnt/data") / "00_Memo" / INPUT_XLSX_NAME,
    Path("/mnt/data") / INPUT_XLSX_NAME,
]

csv_path = next((p for p in candidate_csv_paths if p.exists()), None)
xlsx_path = next((p for p in candidate_xlsx_paths if p.exists()), None)

print("Current working directory:", cwd)

if csv_path is None and xlsx_path is None:
    print("\nSearched CSV paths:")
    for p in candidate_csv_paths:
        print(" -", p)

    print("\nSearched XLSX paths:")
    for p in candidate_xlsx_paths:
        print(" -", p)

    raise FileNotFoundError(
        "Could not find the input CSV or XLSX. "
        "Check that the files are inside 00_Memo or the same folder as this notebook."
    )

if csv_path is not None:
    df = pd.read_csv(csv_path, encoding="utf-8-sig")
    source_path = csv_path
else:
    try:
        df = pd.read_excel(xlsx_path, sheet_name="Full_Literature_Table")
    except ValueError:
        df = pd.read_excel(xlsx_path)
    source_path = xlsx_path

print(f"\nLoaded: {source_path}")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

display(df.head(3))

Current working directory: c:\Users\junse\Documents\research\AI_Search_GEO_Literature_Review\09_code

Loaded: c:\Users\junse\Documents\research\AI_Search_GEO_Literature_Review\00_Memo\GEO_AI_Search_Literature_Review_Table_FINAL_metrics_updated.csv
Shape: 23 rows × 47 columns


,Paper Title,Authors,Year,Venue / Status,Institution / Affiliation,Topic Category,Paper Type,Main Research Question,Key Findings,Data Source(s),AI Systems / Platforms Studied,Outcome Variables,Methods,Benchmark / Evaluation Dataset,Dataset Composition,Primary Metric,Secondary Metrics,Reported Main Score,Metric Category,Higher-is-Better?,Evaluation Unit,Code / GitHub Available?,Dataset Available?,Paper Link,GitHub Link,Dataset Link,Relevance for Finance/Econ Project,Limitations / Open Gaps,Priority Label,Priority Score,Priority Notes,Read Status,Summary Status,Verification Level,Verification Notes,PDF File Name,Folder Code,Recommended Folder,Folder Display Name,Suggested Relative Path,Folder Sort Key,Folder Role / Scope,Primary Review Function,Secondary Tags,Organization Memo,Move / Rename Status,Notes
0,GEO: Generative Engine Optimization,Pranjal Aggarwal; Vishvak Murahari; Tanmay Rajpurohit; Ashwin Kalyan; Karthik Narasimhan; Ameet Deshpande,2024,"KDD '24 / Published in Proceedings of the 30th ACM SIGKDD Conference on Knowledge Discovery and Data Mining, August ...",Indian Institute of Technology Delhi; Princeton University; Independent researchers based in Seattle,Generative Engine Optimization; AI search visibility; website optimization for generative AI systems; information re...,Empirical benchmark + framework paper,How can website and content creators optimize their web content to improve visibility in generative engine responses...,The paper formalizes Generative Engine Optimization (GEO) as a creator-centric optimization paradigm for AI-mediated...,"GEO-bench: 10,000 queries from MS MARCO, ORCAS-I, Natural Questions, AllSouls, LIMA, Davinci-Debate, Perplexity.ai D...",GPT-3.5-turbo-based simulated generative engine; Google Search for source retrieval; Perplexity.ai as a deployed rea...,Position-Adjusted Word Count; Word Count; citation position; Subjective Impression; relevance; influence; uniqueness...,Black-box optimization framework for modifying website content. The authors apply nine GEO methods to source website...,GEO-bench; additional in-the-wild Perplexity.ai evaluation,"10K queries split into 8K train, 1K validation, and 1K test; nine query sources including MS MARCO, ORCAS-I, Natural...",Position-Adjusted Word Count visibility / impression score,Word Count; Position Count; Subjective Impression; relevance; influence; uniqueness; diversity; follow-up/click like...,Best GEO methods improve over baseline by 41% on Position-Adjusted Word Count and 28% on Subjective Impression on GE...,Citation visibility / generative-engine impression / GEO optimization effectiveness,Yes,Optimized source website/citation within a query-level generative-engine response,Yes. The PDF states that code and data are available at the project page and mentions a public repository for prompt...,Yes. The paper releases GEO-bench and states that code and data are available at the project page.,https://doi.org/10.1145/3637528.3671900,https://github.com/GEO-optim/GEO,https://generative-engines.com/GEO/,Highly relevant. This paper directly supports a finance/econ project studying how firms may adapt website content an...,"The paper tests only a limited number of generative engine settings, so methods may need to adapt as generative engi...",High,10.0,Must Read,Read,Summarized,Verified,"From uploaded PDF. Main claims checked against the paper text, including title/authors/venue, GEO-bench design, GEO ...",1_2024_Aggarwal_Generative_Engine_Optimization.pdf,2,02_Core_GEO_AI_Visibility,Core GEO & AI Visibility,GEO_AI_Search_Literature_Review/02_Core_GEO_AI_Visibility/1_2024_Aggarwal_Generative_Engine_Optimization.pdf,02-001,"Foundational and applied GEO papers about improving AI-search visibility, citation exposure, and generative-engine r...",Anchor paper for defining GEO and motivating the literature review.,GEO; AI visibility; content optimization; generative search,Place this paper in 02_Core_GEO_AI_Visibility. Keep the PDF filename as 1_2024

## 2. Clean paper IDs and apply score-based priority categories

This step explicitly ignores the original `Priority Label` field and derives all priority categories from `Priority Score`.

In [7]:
def extract_paper_id(row, fallback_index):
    # Extract leading paper number from PDF filename; fallback to row order.
    pdf = str(row.get("PDF File Name", ""))
    m = re.match(r"^\s*(\d+)_", pdf)
    if m:
        return int(m.group(1))
    return int(fallback_index + 1)

df = df.copy()
df["paper_id"] = [extract_paper_id(row, i) for i, row in df.iterrows()]
df = df.sort_values("paper_id").reset_index(drop=True)

df["Priority Score"] = pd.to_numeric(df["Priority Score"], errors="coerce")

def score_meaning(score):
    if pd.isna(score):
        return "Missing score"
    if 9 <= score <= 10:
        return "Must-read / central paper"
    if 8 <= score < 9:
        return "Highly relevant, should include in memo"
    if 7 <= score < 8:
        return "Useful supporting paper"
    if 6 <= score < 7:
        return "Background only"
    return "Probably mention briefly or exclude"

def score_role(score):
    if pd.isna(score):
        return "Unscored"
    if 9 <= score <= 10:
        return "Central"
    if 8 <= score < 9:
        return "Main-body"
    if 7 <= score < 8:
        return "Supporting"
    if 6 <= score < 7:
        return "Background"
    return "Brief/Exclude"

df["Score Meaning"] = df["Priority Score"].apply(score_meaning)
df["Score-Based Role"] = df["Priority Score"].apply(score_role)

priority_counts = df["Score Meaning"].value_counts().rename_axis("Score Meaning").reset_index(name="Paper Count")
display(priority_counts)
display(df[["paper_id", "Paper Title", "Priority Score", "Score Meaning", "Score-Based Role"]])

,Score Meaning,Paper Count
0,Must-read / central paper,10
1,"Highly relevant, should include in memo",10
2,Useful supporting paper,3


,paper_id,Paper Title,Priority Score,Score Meaning,Score-Based Role
0,1,GEO: Generative Engine Optimization,10.0,Must-read / central paper,Central
1,2,SAGEO Arena: A Realistic Environment for Evaluating Search-Augmented Generative Engine Optimization,9.5,Must-read / central paper,Central
2,3,From Citation Selection to Citation Absorption: A Measurement Framework for Generative Engine Optimization Across AI...,9.0,Must-read / central paper,Central
3,4,Think Before Writing: Feature-Level Multi-Objective Optimization for Generative Citation Visibility,8.5,"Highly relevant, should include in memo",Main-body
4,5,AgenticGEO: A Self-Evolving Agentic System for Generative Engine Optimization,9.0,Must-read / central paper,Central
5,6,Structural Feature Engineering for Generative Engine Optimization: How Content Structure Shapes Citation Behavior,8.0,"Highly relevant, should include in memo",Main-body
6,7,EcoGEO: Trajectory-Aware Evidence Ecosystems for Web-Enabled LLM Search Agents,8.5,"Highly relevant, should include in memo",Main-body
7,8,What Generative Search Engines Like and How to Optimize Web Content Cooperatively,10.0,Must-read / central paper,Central
8,9,E-GEO: A Testbed for Generative Engine Optimization in E-Commerce,9.5,Must-read / central paper,Central
9,10,"How Generative AI Disrupts Search: An Empirical Study of Google Search, Gemini, and AI Overviews",9.5,Must-read / central paper,Central


## 3. Manual comparative taxonomy

The mappings below are intentionally transparent and editable. They convert the literature table into an analysis-ready structure.

The taxonomy is not meant to replace the original paper-level table; it is a higher-level coding layer for comparison, similarity, and limitation analysis.

In [8]:
SHORT_NAMES = {
    1: "GEO",
    2: "SAGEO Arena",
    3: "Citation Selection→Absorption",
    4: "FeatGEO",
    5: "AgenticGEO",
    6: "GEO-SFE",
    7: "EcoGEO",
    8: "AutoGEO",
    9: "E-GEO",
    10: "AI Disrupts Search",
    11: "Google AIO Measurement",
    12: "Search Arena",
    13: "News Citing Patterns",
    14: "Google AIO/FS Audit",
    15: "MMSearch",
    16: "FRAMES",
    17: "MIRAGE",
    18: "LaRA",
    19: "REAL-MM-RAG",
    20: "FinRAGBench-V",
    21: "HOH",
    22: "DailyQA",
    23: "ALCE",
}

EVIDENCE_TYPE = {
    1: "GEO optimization / website adaptation",
    2: "Full-pipeline SAGEO / structured search",
    3: "AI visibility measurement / citation absorption",
    4: "Feature-level GEO optimization",
    5: "Agentic GEO optimization",
    6: "Structured content / website optimization",
    7: "Ecosystem / agentic web-search GEO",
    8: "Preference-rule GEO optimization",
    9: "E-commerce GEO / product ranking",
    10: "Live AI search platform audit",
    11: "Live AI search platform audit",
    12: "Search-augmented LLM user preference",
    13: "AI search source gatekeeping",
    14: "Prompt-based AI search audit",
    15: "Multimodal AI search benchmark",
    16: "RAG reasoning benchmark",
    17: "RAG robustness benchmark",
    18: "RAG vs long-context benchmark",
    19: "Multimodal retrieval benchmark",
    20: "Finance multimodal RAG benchmark",
    21: "Temporal RAG / outdated information",
    22: "Dynamic web RAG / real-world changes",
    23: "Citation quality / attributed generation",
}

BENCHMARK_TYPE = {
    1: "GEO optimization benchmark",
    2: "Full-pipeline search benchmark",
    3: "Citation measurement dataset",
    4: "Feature-level optimization benchmark",
    5: "Agentic optimization benchmark",
    6: "Structural optimization benchmark",
    7: "Controlled agentic/evidence-ecosystem benchmark",
    8: "Preference-rule optimization benchmark",
    9: "E-commerce GEO benchmark",
    10: "Live platform audit",
    11: "Live platform audit",
    12: "Crowd-sourced preference benchmark",
    13: "Citation-pattern audit",
    14: "Prompt-based audit",
    15: "Multimodal search benchmark",
    16: "RAG evaluation benchmark",
    17: "RAG evaluation benchmark",
    18: "RAG vs long-context benchmark",
    19: "Multimodal retrieval benchmark",
    20: "Finance multimodal RAG benchmark",
    21: "Temporal RAG benchmark",
    22: "Dynamic web RAG benchmark",
    23: "Citation evaluation benchmark",
}

METRIC_FAMILY = {
    1: "AI visibility / impression",
    2: "Stage-level search visibility",
    3: "Citation selection and absorption",
    4: "Visibility-quality multi-objective optimization",
    5: "Adaptive GEO visibility gain",
    6: "Structural visibility / citation rate",
    7: "Recommendation / trajectory influence",
    8: "GEO visibility + generative-engine utility",
    9: "Product ranking / recommendation visibility",
    10: "Source overlap / AIO activation / robustness",
    11: "AIO activation / source quality / claim fidelity",
    12: "User preference / citation features",
    13: "Source concentration / news citation bias",
    14: "AIO-FS consistency / credibility audit",
    15: "Multimodal search accuracy",
    16: "RAG accuracy / multi-hop reasoning",
    17: "RAG robustness / noise-context metrics",
    18: "RAG vs long-context QA accuracy",
    19: "Multimodal retrieval performance",
    20: "Financial RAG retrieval + visual citation",
    21: "Outdated-information harm / RAG accuracy",
    22: "Date-specific accuracy / temporal web retrieval",
    23: "Citation precision/recall / support quality",
}

EVALUATION_LEVEL = {
    1: "Source in generated answer",
    2: "Target document across retrieval-reranking-generation",
    3: "Prompt-platform pair and fetched citation page",
    4: "Webpage feature configuration",
    5: "Document strategy/query",
    6: "Webpage structural variant",
    7: "Query-product/evidence ecosystem",
    8: "Query-document rewrite",
    9: "Product listing/rank",
    10: "Query-level source set",
    11: "AIO/source/claim",
    12: "Conversation/response/user vote",
    13: "Response/news citation",
    14: "Query/AIO/Featured Snippet pair",
    15: "Multimodal query/search stage",
    16: "Question/answer",
    17: "QA pair/context condition",
    18: "QA over long context",
    19: "Query-page/document retrieval",
    20: "Financial QA/page/block citation",
    21: "Time-sensitive QA/retrieved evidence",
    22: "Daily/weekly updated query",
    23: "Answer statement/citation",
}

FINANCE_RELEVANCE = {
    1: "Medium",
    2: "High",
    3: "High",
    4: "Medium",
    5: "Medium",
    6: "Medium",
    7: "Medium",
    8: "High",
    9: "High",
    10: "High",
    11: "High",
    12: "Medium",
    13: "Medium",
    14: "Medium",
    15: "Low-Medium",
    16: "Low-Medium",
    17: "Low-Medium",
    18: "High",
    19: "High",
    20: "Very High",
    21: "High",
    22: "High",
    23: "High",
}

df["Short Name"] = df["paper_id"].map(SHORT_NAMES)
df["Evidence Type"] = df["paper_id"].map(EVIDENCE_TYPE)
df["Benchmark Type"] = df["paper_id"].map(BENCHMARK_TYPE)
df["Metric Family"] = df["paper_id"].map(METRIC_FAMILY)
df["Evaluation Level"] = df["paper_id"].map(EVALUATION_LEVEL)
df["Finance Relevance Level"] = df["paper_id"].map(FINANCE_RELEVANCE)

display(df[["paper_id", "Short Name", "Priority Score", "Score Meaning", "Evidence Type", "Metric Family", "Finance Relevance Level"]])

,paper_id,Short Name,Priority Score,Score Meaning,Evidence Type,Metric Family,Finance Relevance Level
0,1,GEO,10.0,Must-read / central paper,GEO optimization / website adaptation,AI visibility / impression,Medium
1,2,SAGEO Arena,9.5,Must-read / central paper,Full-pipeline SAGEO / structured search,Stage-level search visibility,High
2,3,Citation Selection→Absorption,9.0,Must-read / central paper,AI visibility measurement / citation absorption,Citation selection and absorption,High
3,4,FeatGEO,8.5,"Highly relevant, should include in memo",Feature-level GEO optimization,Visibility-quality multi-objective optimization,Medium
4,5,AgenticGEO,9.0,Must-read / central paper,Agentic GEO optimization,Adaptive GEO visibility gain,Medium
5,6,GEO-SFE,8.0,"Highly relevant, should include in memo",Structured content / website optimization,Structural visibility / citation rate,Medium
6,7,EcoGEO,8.5,"Highly relevant, should include in memo",Ecosystem / agentic web-search GEO,Recommendation / trajectory influence,Medium
7,8,AutoGEO,10.0,Must-read / central paper,Preference-rule GEO optimization,GEO visibility + generative-engine utility,High
8,9,E-GEO,9.5,Must-read / central paper,E-commerce GEO / product ranking,Product ranking / recommendation visibility,High
9,10,AI Disrupts Search,9.5,Must-read / central paper,Live AI search platform audit,Source overlap / AIO activation / robustness,High


## 4. Build paper taxonomy table

This table is the first worksheet in the comparative analysis workbook. It uses the score-based meaning rather than the original `Priority Label`.

In [9]:
taxonomy_cols = [
    "paper_id", "Short Name", "Paper Title", "Authors", "Year", "Venue / Status",
    "Priority Score", "Score Meaning", "Score-Based Role",
    "Evidence Type", "Benchmark Type", "Metric Family", "Evaluation Level", "Finance Relevance Level",
    "Recommended Folder", "Primary Review Function", "Secondary Tags",
]

taxonomy = df[taxonomy_cols].copy()
taxonomy = taxonomy.rename(columns={
    "paper_id": "Paper No.",
    "Priority Score": "Score",
})

display(taxonomy.head())

,Paper No.,Short Name,Paper Title,Authors,Year,Venue / Status,Score,Score Meaning,Score-Based Role,Evidence Type,Benchmark Type,Metric Family,Evaluation Level,Finance Relevance Level,Recommended Folder,Primary Review Function,Secondary Tags
0,1,GEO,GEO: Generative Engine Optimization,Pranjal Aggarwal; Vishvak Murahari; Tanmay Rajpurohit; Ashwin Kalyan; Karthik Narasimhan; Ameet Deshpande,2024,"KDD '24 / Published in Proceedings of the 30th ACM SIGKDD Conference on Knowledge Discovery and Data Mining, August ...",10.0,Must-read / central paper,Central,GEO optimization / website adaptation,GEO optimization benchmark,AI visibility / impression,Source in generated answer,Medium,02_Core_GEO_AI_Visibility,Anchor paper for defining GEO and motivating the literature review.,GEO; AI visibility; content optimization; generative search
1,2,SAGEO Arena,SAGEO Arena: A Realistic Environment for Evaluating Search-Augmented Generative Engine Optimization,Sunghwan Kim; Wooseok Jeong; Serin Kim; Sangam Lee; Dongha Lee,2026,"KDD 2026 / arXiv preprint, Work in Progress; venue listed on author page, while the arXiv version is marked Work in ...",9.5,Must-read / central paper,Central,Full-pipeline SAGEO / structured search,Full-pipeline search benchmark,Stage-level search visibility,Target document across retrieval-reranking-generation,High,02_Core_GEO_AI_Visibility,"Use to explain full-pipeline SAGEO: retrieval, reranking, and generation.",GEO; SAGEO; search-augmented optimization; retrieval-reranking-generation
2,3,Citation Selection→Absorption,From Citation Selection to Citation Absorption: A Measurement Framework for Generative Engine Optimization Across AI...,Zhang Kai; He Xinyue; Yao Jingang,2026,arXiv preprint v2 / empirical measurement framework; no peer-reviewed conference venue verified,9.0,Must-read / central paper,Central,AI visibility measurement / citation absorption,Citation measurement dataset,Citation selection and absorption,Prompt-platform pair and fetched citation page,High,06_Auditing_Benchmarking,Use as the GEO measurement framework: citation selection versus citation absorption.,GEO measurement; citation selection; citation absorption; AI search platforms
3,4,FeatGEO,Think Before Writing: Feature-Level Multi-Objective Optimization for Generative Citation Visibility,Zikang Liu; Peilan Xu,2026,"arXiv preprint; GitHub README states accepted to ACL 2026, but official proceedings/listing should be verified later",8.5,"Highly relevant, should include in memo",Main-body,Feature-level GEO optimization,Feature-level optimization benchmark,Visibility-quality multi-objective optimization,Webpage feature configuration,Medium,04_Structured_Content_Schema_Web,Use as the feature-level optimization paper for citation visibility.,Feature-level optimization; citation visibility; structured writing
4,5,AgenticGEO,AgenticGEO: A Self-Evolving Agentic System for Generative Engine Optimization,Jiaqi Yuan; Jialu Wang; Zihan Wang; Qingyun Sun; Ruijie Wang; Jianxin Li,2026,arXiv preprint; ACM-style template contains venue placeholders; code and model are publicly available,9.0,Must-read / central paper,Central,Agentic GEO optimization,Agentic optimization benchmark,Adaptive GEO visibility gain,Document strategy/query,Medium,02_Core_GEO_AI_Visibility,Use as the agentic / self-evolving GEO method paper.,Agentic GEO; autonomous optimization; AI visibility


## 5. Benchmark comparison table

This sheet makes clear that headline results are not directly comparable across all papers because papers use different evaluation units: source, target document, product, claim, page, answer, user vote, or date-specific query.

In [10]:
benchmark_cols = [
    "paper_id", "Short Name", "Paper Title", "Benchmark / Evaluation Dataset",
    "Dataset Composition", "Benchmark Type", "AI Systems / Platforms Studied",
    "Evaluation Level", "Primary Metric", "Secondary Metrics", "Metric Family",
    "Reported Main Score", "Higher-is-Better?", "Relevance for Finance/Econ Project",
    "Limitations / Open Gaps"
]

benchmark_comparison = df[benchmark_cols].copy()
benchmark_comparison = benchmark_comparison.rename(columns={"paper_id": "Paper No."})

display(benchmark_comparison.head(8))

,Paper No.,Short Name,Paper Title,Benchmark / Evaluation Dataset,Dataset Composition,Benchmark Type,AI Systems / Platforms Studied,Evaluation Level,Primary Metric,Secondary Metrics,Metric Family,Reported Main Score,Higher-is-Better?,Relevance for Finance/Econ Project,Limitations / Open Gaps
0,1,GEO,GEO: Generative Engine Optimization,GEO-bench; additional in-the-wild Perplexity.ai evaluation,"10K queries split into 8K train, 1K validation, and 1K test; nine query sources including MS MARCO, ORCAS-I, Natural...",GEO optimization benchmark,GPT-3.5-turbo-based simulated generative engine; Google Search for source retrieval; Perplexity.ai as a deployed rea...,Source in generated answer,Position-Adjusted Word Count visibility / impression score,Word Count; Position Count; Subjective Impression; relevance; influence; uniqueness; diversity; follow-up/click like...,AI visibility / impression,Best GEO methods improve over baseline by 41% on Position-Adjusted Word Count and 28% on Subjective Impression on GE...,Yes,Highly relevant. This paper directly supports a finance/econ project studying how firms may adapt website content an...,"The paper tests only a limited number of generative engine settings, so methods may need to adapt as generative engi..."
1,2,SAGEO Arena,SAGEO Arena: A Realistic Environment for Evaluating Search-Augmented Generative Engine Optimization,SAGEO Arena,"2,700 sampled queries and 171,003 retrieved documents across nine domains/datasets: MS MARCO/Web Search, Natural Que...",Full-pipeline search benchmark,"A simulated search-augmented generative engine pipeline with BM25 retrieval, cross-encoder reranking, and LLM genera...",Target document across retrieval-reranking-generation,"Stage-level target-document visibility measured by Hit Rate and Rank Change across retrieval, reranking, and generation",Retrieval-stage hit/rank change; reranking-stage hit/rank change; generation-stage citation/citation-rate change; do...,Stage-level search visibility,No single scalar headline score; the paper reports that stage-aware SAGEO achieves the strongest/most competitive ov...,Mixed: higher Hit Rate and generation citation rate are better; Rank Change is better when the optimized document mo...,"Highly relevant. This paper is directly useful for a finance/econ project on how firms may adapt website content, me...","The paper uses a constructed benchmark rather than live commercial systems such as ChatGPT, Perplexity, Gemini, or G..."
2,3,Citation Selection→Absorption,From Citation Selection to Citation Absorption: A Measurement Framework for Generative Engine Optimization Across AI...,geo-citation-lab public dataset and report,"602 controlled prompts across ChatGPT, Google AI Overview/Gemini, and Perplexity; prompt layers A/B/C/D = 432/60/60/...",Citation measurement dataset,ChatGPT; Google AI Overview/Gemini; Perplexity,Prompt-platform pair and fetched citation page,Citation absorption influence_score; citation count/citation breadth for the selection stage,Search trigger rate; mean/median/max citations per prompt; fetch-ok rate; mean/median influence by platform; page wo...,Citation selection and absorption,"Mean citations per prompt: ChatGPT 6.88, Google 12.06, Perplexity 16.35. Mean fetched-page influence: ChatGPT 0.2713...","Mostly yes for citation count, selection rate, and influence_score, but the paper emphasizes that breadth and absorp...",Highly relevant. The paper provides measurable dependent variables for AI-mediated visibility beyond traditional SEO...,The paper is primarily descriptive and does not establish causal effects of page features on citation outcomes. The ...
3,4,FeatGEO,Think Before Writing: Feature-Level Multi-Objective Optimization for Generative Citation Visibility,GEO-Bench,"10K GEO-Bench queries spanning 25 domains from nine sources; for each query, an advertiser-controlled page is inject...",Feature-level optimization benchmark,"Controlled generative-answer pipeline with GPT-4o-mini, Gemini-2.5-fl

## 6. Metric crosswalk

This table translates CS metrics into finance/econ-relevant outcome families.

In [11]:
metric_crosswalk = pd.DataFrame([
    {
        "Metric Family": "AI visibility / impression",
        "Example Metrics": "Position-adjusted word count; impression score; visibility score; citation rate",
        "Relevant Papers": "1, 4, 5, 6, 8",
        "Finance/Econ Interpretation": "How visible a firm, source, product, or publisher is inside an AI-generated answer.",
        "Possible Dependent Variable": "AI visibility score; answer inclusion; citation rate; citation position",
    },
    {
        "Metric Family": "Stage-level search visibility",
        "Example Metrics": "Retrieval hit rate; reranking hit rate; citation rate; rank change",
        "Relevant Papers": "2",
        "Finance/Econ Interpretation": "Where a firm/source drops out of the AI search pipeline.",
        "Possible Dependent Variable": "retrieval inclusion; rerank position; generation citation",
    },
    {
        "Metric Family": "Product / recommendation ranking",
        "Example Metrics": "Product rank; rank change; recommendation probability; target inclusion",
        "Relevant Papers": "7, 9",
        "Finance/Econ Interpretation": "AI-mediated product discovery and seller competition.",
        "Possible Dependent Variable": "product rank under AI shopping/search prompts",
    },
    {
        "Metric Family": "Source selection / overlap",
        "Example Metrics": "Citation count; Jaccard similarity; rank-biased overlap; source concentration",
        "Relevant Papers": "3, 10, 11, 13, 14",
        "Finance/Econ Interpretation": "Whether AI search reallocates visibility relative to traditional search or concentrates attention among sources.",
        "Possible Dependent Variable": "AIO/SERP overlap; cross-platform source overlap; source concentration index",
    },
    {
        "Metric Family": "Citation absorption / influence",
        "Example Metrics": "Influence score; answer-source semantic overlap; first citation position; paragraph coverage",
        "Relevant Papers": "3",
        "Finance/Econ Interpretation": "Whether a cited source actually shapes the generated answer.",
        "Possible Dependent Variable": "source absorption score; answer influence score",
    },
    {
        "Metric Family": "Citation fidelity / support quality",
        "Example Metrics": "Citation precision; citation recall; unsupported-claim rate; contradiction rate",
        "Relevant Papers": "11, 20, 23",
        "Finance/Econ Interpretation": "Whether cited financial or consumer claims are supported by the underlying source.",
        "Possible Dependent Variable": "claim support rate; unsupported financial claim rate",
    },
    {
        "Metric Family": "User preference",
        "Example Metrics": "Bradley-Terry preference score; user votes; citation/length coefficients",
        "Relevant Papers": "12, 13",
        "Finance/Econ Interpretation": "Whether source presentation and citation style affect perceived credibility and user choice.",
        "Possible Dependent Variable": "user preference for AI answer; vote share",
    },
    {
        "Metric Family": "RAG accuracy / robustness",
        "Example Metrics": "Accuracy; Recall@k; nDCG; context acceptability; noise vulnerability",
        "Relevant Papers": "16, 17, 18, 19, 20",
        "Finance/Econ Interpretation": "Whether AI systems can retrieve and use correct evidence from long or noisy firm/financial documents.",
        "Possible Dependent Variable": "retrieval accuracy; answer accuracy; context robustness score",
    },
    {
        "Metric Family": "Temporal freshness",
        "Example Metrics": "Outdated-information harm; date-specific accuracy; update-frequency accuracy",
        "Relevant Papers": "21, 22",
        "Finance/Econ Interpretation": "Whether AI search uses current rather than stale firm, market, regulatory, or macroeconomic facts.",
        "Possible Dependent Variable": "outdated answer rate; date-specific correctness",
    },
    {
        "Metric Family": "Visual citation / multimodal evidence",
        "Example Metrics": "Page-level citation precision/recall; block-level citation precision/recall; multimodal retrieval recall",
        "Relevant Papers": "19, 20",
        "Finance/Econ Interpretation": "Whether AI systems can cite correct pages, tables, charts, and blocks in financial disclosures.",
        "Possible Dependent Variable": "visual citation precision/recall for financial reports",
    },
])

display(metric_crosswalk)

,Metric Family,Example Metrics,Relevant Papers,Finance/Econ Interpretation,Possible Dependent Variable
0,AI visibility / impression,Position-adjusted word count; impression score; visibility score; citation rate,"1, 4, 5, 6, 8","How visible a firm, source, product, or publisher is inside an AI-generated answer.",AI visibility score; answer inclusion; citation rate; citation position
1,Stage-level search visibility,Retrieval hit rate; reranking hit rate; citation rate; rank change,2,Where a firm/source drops out of the AI search pipeline.,retrieval inclusion; rerank position; generation citation
2,Product / recommendation ranking,Product rank; rank change; recommendation probability; target inclusion,"7, 9",AI-mediated product discovery and seller competition.,product rank under AI shopping/search prompts
3,Source selection / overlap,Citation count; Jaccard similarity; rank-biased overlap; source concentration,"3, 10, 11, 13, 14",Whether AI search reallocates visibility relative to traditional search or concentrates attention among sources.,AIO/SERP overlap; cross-platform source overlap; source concentration index
4,Citation absorption / influence,Influence score; answer-source semantic overlap; first citation position; paragraph coverage,3,Whether a cited source actually shapes the generated answer.,source absorption score; answer influence score
5,Citation fidelity / support quality,Citation precision; citation recall; unsupported-claim rate; contradiction rate,"11, 20, 23",Whether cited financial or consumer claims are supported by the underlying source.,claim support rate; unsupported financial claim rate
6,User preference,Bradley-Terry preference score; user votes; citation/length coefficients,"12, 13",Whether source presentation and citation style affect perceived credibility and user choice.,user preference for AI answer; vote share
7,RAG accuracy / robustness,Accuracy; Recall@k; nDCG; context acceptability; noise vulnerability,"16, 17, 18, 19, 20",Whether AI systems can retrieve and use correct evidence from long or noisy firm/financial documents.,retrieval accuracy; answer accuracy; context robustness score
8,Temporal freshness,Outdated-information harm; date-specific accuracy; update-frequency accuracy,"21, 22","Whether AI search uses current rather than stale firm, market, regulatory, or macroeconomic facts.",outdated answer rate; date-specific correctness
9,Visual citation / multimodal evidence,Page-level citation precision/recall; block-level citation precision/recall; multimodal retrieval recall,"19, 20","Whether AI systems can cite correct pages, tables, charts, and blocks in financial disclosures.",visual citation precision/recall for financial reports


## 7. Similarity feature matrix

Feature coding uses 0 / 0.5 / 1 values:
- `0` = not a meaningful focus
- `0.5` = partial or indirect focus
- `1` = central focus

These values are manually coded from the paper summaries and are meant to be transparent and editable.

In [12]:
FEATURES = [
    "GEO_Optimization",
    "Stage_Aware_Search",
    "Structured_Content",
    "Citation_Measurement",
    "Citation_Fidelity",
    "Live_Platform_Audit",
    "RAG_Evaluation",
    "Multimodal",
    "Finance_Specific",
    "Ecommerce_Product",
    "User_Preference",
    "Temporal_Freshness",
    "Public_Code_Data",
    "Economic_Outcome_Proxy",
]

# Manual feature coding: 0 = no, 0.5 = partial, 1 = yes
FEATURE_VALUES = {
    1:  [1,   0,   0.5, 1,   0,   0.5, 0.5, 0,   0,   0,   0,   0,   1,   0.5],
    2:  [1,   1,   1,   1,   0,   0,   1,   0,   0.5, 0.5, 0,   0,   0,   0.5],
    3:  [0,   0,   0.5, 1,   0.5, 1,   0.5, 0,   0.5, 0,   0,   0,   1,   0.5],
    4:  [1,   0,   1,   1,   0.5, 0,   0.5, 0,   0,   0,   0,   0,   1,   0.5],
    5:  [1,   0,   0.5, 1,   0,   0,   0.5, 0,   0,   0.5, 0,   0,   1,   0.5],
    6:  [1,   0,   1,   1,   0,   0,   0.5, 0,   0,   0,   0,   0,   0,   0.5],
    7:  [1,   1,   1,   0.5, 0,   0,   0.5, 0,   0,   1,   0,   0,   0,   1],
    8:  [1,   0.5, 0.5, 1,   0.5, 0,   0.5, 0,   0,   0.5, 0,   0,   1,   0.5],
    9:  [1,   0,   0.5, 0.5, 0,   0,   0.5, 0,   0,   1,   0,   0,   1,   1],
    10: [0,   0.5, 0,   1,   0.5, 1,   0.5, 0,   0,   0.5, 0,   0,   1,   1],
    11: [0,   0,   0,   1,   1,   1,   0.5, 0,   0,   0,   0,   0,   0.5, 1],
    12: [0,   0,   0,   1,   0.5, 0.5, 0.5, 0,   0,   0,   1,   0,   1,   0.5],
    13: [0,   0,   0,   1,   0.5, 0.5, 0.5, 0,   0,   0,   1,   0,   1,   0.5],
    14: [0,   0,   0,   1,   0.5, 1,   0.5, 0,   0,   0,   0,   0,   1,   0.5],
    15: [0,   0.5, 0,   0,   0,   0,   0.5, 1,   0,   0,   0,   0,   1,   0],
    16: [0,   0,   0,   0,   0,   0,   1,   0,   0,   0,   0,   0,   1,   0],
    17: [0,   0,   0,   0.5, 0.5, 0,   1,   0,   0,   0,   0,   0,   1,   0],
    18: [0,   0,   0,   0,   0,   0,   1,   0,   1,   0,   0,   0,   1,   0.5],
    19: [0,   0,   0,   0,   0,   0,   1,   1,   1,   0,   0,   0,   1,   0.5],
    20: [0,   0,   0,   1,   1,   0,   1,   1,   1,   0,   0,   0,   1,   0.5],
    21: [0,   0,   0,   0,   0.5, 0,   1,   0,   0.5, 0,   0,   1,   1,   0.5],
    22: [0,   0,   0,   0,   0,   0,   1,   0,   0.5, 0,   0,   1,   0.5, 0.5],
    23: [0,   0,   0,   1,   1,   0,   1,   0,   0,   0,   0,   0,   1,   0],
}

features_df = pd.DataFrame.from_dict(FEATURE_VALUES, orient="index", columns=FEATURES)
features_df.index.name = "Paper No."
features_df = features_df.reset_index()
features_df.insert(1, "Short Name", features_df["Paper No."].map(SHORT_NAMES))
features_df.insert(2, "Score", features_df["Paper No."].map(df.set_index("paper_id")["Priority Score"]))
features_df.insert(3, "Score Meaning", features_df["Paper No."].map(df.set_index("paper_id")["Score Meaning"]))

display(features_df)

,Paper No.,Short Name,Score,Score Meaning,GEO_Optimization,Stage_Aware_Search,Structured_Content,Citation_Measurement,Citation_Fidelity,Live_Platform_Audit,RAG_Evaluation,Multimodal,Finance_Specific,Ecommerce_Product,User_Preference,Temporal_Freshness,Public_Code_Data,Economic_Outcome_Proxy
0,1,GEO,10.0,Must-read / central paper,1,0.0,0.5,1.0,0.0,0.5,0.5,0,0.0,0.0,0,0,1.0,0.5
1,2,SAGEO Arena,9.5,Must-read / central paper,1,1.0,1.0,1.0,0.0,0.0,1.0,0,0.5,0.5,0,0,0.0,0.5
2,3,Citation Selection→Absorption,9.0,Must-read / central paper,0,0.0,0.5,1.0,0.5,1.0,0.5,0,0.5,0.0,0,0,1.0,0.5
3,4,FeatGEO,8.5,"Highly relevant, should include in memo",1,0.0,1.0,1.0,0.5,0.0,0.5,0,0.0,0.0,0,0,1.0,0.5
4,5,AgenticGEO,9.0,Must-read / central paper,1,0.0,0.5,1.0,0.0,0.0,0.5,0,0.0,0.5,0,0,1.0,0.5
5,6,GEO-SFE,8.0,"Highly relevant, should include in memo",1,0.0,1.0,1.0,0.0,0.0,0.5,0,0.0,0.0,0,0,0.0,0.5
6,7,EcoGEO,8.5,"Highly relevant, should include in memo",1,1.0,1.0,0.5,0.0,0.0,0.5,0,0.0,1.0,0,0,0.0,1.0
7,8,AutoGEO,10.0,Must-read / central paper,1,0.5,0.5,1.0,0.5,0.0,0.5,0,0.0,0.5,0,0,1.0,0.5
8,9,E-GEO,9.5,Must-read / central paper,1,0.0,0.5,0.5,0.0,0.0,0.5,0,0.0,1.0,0,0,1.0,1.0
9,10,AI Disrupts Search,9.5,Must-read / central paper,0,0.5,0.0,1.0,0.5,1.0,0.5,0,0.0,0.5,0,0,1.0,1.0


## 8. Paper-pair similarity matrix

Cosine similarity is computed over the manually coded feature matrix. The goal is not to claim exact mathematical truth; it is to create a transparent, reproducible map of conceptual similarity across papers.

In [13]:
feature_matrix = features_df[FEATURES].to_numpy(dtype=float)
norms = np.linalg.norm(feature_matrix, axis=1, keepdims=True)
norms[norms == 0] = 1
normed = feature_matrix / norms
similarity = normed @ normed.T

labels = [f"{int(row['Paper No.']):02d} {row['Short Name']}" for _, row in features_df.iterrows()]
similarity_df = pd.DataFrame(similarity, index=labels, columns=labels)

display(similarity_df.round(2))

,01 GEO,02 SAGEO Arena,03 Citation Selection→Absorption,04 FeatGEO,05 AgenticGEO,06 GEO-SFE,07 EcoGEO,08 AutoGEO,09 E-GEO,10 AI Disrupts Search,11 Google AIO Measurement,12 Search Arena,13 News Citing Patterns,14 Google AIO/FS Audit,15 MMSearch,16 FRAMES,17 MIRAGE,18 LaRA,19 REAL-MM-RAG,20 FinRAGBench-V,21 HOH,22 DailyQA,23 ALCE
01 GEO,1.00,0.68,0.79,0.92,0.94,0.80,0.59,0.88,0.80,0.73,0.65,0.69,0.69,0.77,0.40,0.53,0.63,0.49,0.42,0.55,0.45,0.38,0.62
02 SAGEO Arena,0.68,1.00,0.51,0.72,0.73,0.84,0.89,0.79,0.67,0.51,0.39,0.36,0.36,0.38,0.26,0.29,0.40,0.40,0.35,0.46,0.32,0.38,0.42
03 Citation Selection→Absorption,0.79,0.51,1.00,0.72,0.67,0.52,0.36,0.69,0.56,0.87,0.86,0.79,0.79,0.94,0.38,0.51,0.69,0.61,0.53,0.73,0.56,0.44,0.73
04 FeatGEO,0.92,0.72,0.72,1.00,0.92,0.86,0.64,0.92,0.79,0.62,0.59,0.63,0.63,0.65,0.36,0.49,0.65,0.45,0.39,0.60,0.47,0.35,0.69
05 AgenticGEO,0.94,0.73,0.67,0.92,1.00,0.80,0.69,0.94,0.92,0.67,0.53,0.62,0.62,0.65,0.40,0.53,0.63,0.49,0.42,0.55,0.45,0.38,0.62
06 GEO-SFE,0.80,0.84,0.52,0.86,0.80,1.00,0.74,0.76,0.67,0.42,0.44,0.40,0.40,0.41,0.08,0.19,0.34,0.22,0.19,0.37,0.21,0.24,0.40
07 EcoGEO,0.59,0.89,0.36,0.64,0.69,0.74,1.00,0.75,0.78,0.52,0.35,0.27,0.27,0.28,0.20,0.15,0.20,0.24,0.21,0.26,0.22,0.26,0.21
08 AutoGEO,0.88,0.79,0.69,0.92,0.94,0.76,0.75,1.00,0.87,0.74,0.61,0.65,0.65,0.67,0.45,0.50,0.67,0.46,0.40,0.61,0.49,0.36,0.71
09 E-GEO,0.80,0.67,0.56,0.79,0.92,0.67,0.78,0.87,1.00,0.67,0.49,0.52,0.52,0.53,0.36,0.49,0.51,0.51,0.45,0.46,0.47,0.42,0.46
10 AI Disrupts Search,0.73,0.51,0.87,0.62,0.67,0.42,0.52,0.74,0.67,1.00,0.90,0.78,0.78,0.92,0.42,0.47,0.64,0.50,0.43,0.63,0.52,0.40,0.67


## 9. Limitation heatmap coding

Limitation flags use 0 / 0.5 / 1 values:
- `0` = not a major limitation for this paper
- `0.5` = partial limitation
- `1` = major limitation

The limitation heatmap is designed to show where the CS literature leaves space for a finance/econ contribution.

In [14]:
LIMITATIONS = [
    "No_Economic_Outcomes",
    "Simulated_Benchmark",
    "Fixed_Candidate_Set",
    "Observational_NonCausal",
    "LLM_as_Judge",
    "Narrow_Domain",
    "No_Live_Platform",
    "No_Content_Intervention",
    "Platform_Instability",
    "Limited_Multimodal",
    "Limited_Finance_Relevance",
]

LIMITATION_VALUES = {
    1:  [1,   1,   1,   0,   1,   0,   0.5, 0,   0.5, 1,   0.5],
    2:  [1,   1,   0,   0,   1,   0,   1,   0,   0.5, 1,   0],
    3:  [1,   0,   0,   1,   0.5, 0,   0,   1,   0.5, 1,   0],
    4:  [1,   1,   0.5, 0,   1,   0,   1,   0,   0.5, 1,   0.5],
    5:  [1,   1,   0.5, 0,   1,   0,   1,   0,   0.5, 1,   0.5],
    6:  [1,   1,   0.5, 0,   1,   0,   1,   0,   0.5, 1,   0.5],
    7:  [1,   1,   0,   0,   1,   1,   1,   0,   0.5, 1,   0.5],
    8:  [1,   1,   0.5, 0,   1,   0,   1,   0,   0.5, 1,   0.5],
    9:  [0.5, 1,   0.5, 0,   1,   1,   1,   0,   0.5, 1,   0],
    10: [0.5, 0,   0,   1,   0.5, 0,   0,   1,   1,   0.5, 0],
    11: [0.5, 0,   0,   1,   1,   0,   0,   1,   1,   0.5, 0],
    12: [1,   0.5, 0,   1,   0.5, 0,   0,   1,   0.5, 1,   0.5],
    13: [1,   0.5, 0,   1,   0.5, 1,   0,   1,   0.5, 1,   0.5],
    14: [1,   0,   0,   1,   0.5, 1,   0,   1,   1,   0.5, 0.5],
    15: [1,   1,   0,   0,   0.5, 0,   1,   1,   0.5, 0,   1],
    16: [1,   1,   0,   0,   0.5, 0,   1,   1,   0.5, 1,   1],
    17: [1,   1,   0,   0,   0.5, 0,   1,   1,   0.5, 1,   1],
    18: [1,   1,   0,   0,   0.5, 0,   1,   1,   0.5, 0.5, 0],
    19: [1,   1,   0,   0,   0.5, 1,   1,   1,   0.5, 0,   0],
    20: [1,   1,   0,   0,   1,   1,   1,   1,   0.5, 0,   0],
    21: [1,   1,   0,   0,   1,   0,   1,   1,   0.5, 1,   0],
    22: [1,   1,   0,   0,   1,   0,   1,   1,   0.5, 1,   0],
    23: [1,   1,   0.5, 0,   1,   0,   1,   1,   0.5, 1,   0.5],
}

limitations_df = pd.DataFrame.from_dict(LIMITATION_VALUES, orient="index", columns=LIMITATIONS)
limitations_df.index.name = "Paper No."
limitations_df = limitations_df.reset_index()
limitations_df.insert(1, "Short Name", limitations_df["Paper No."].map(SHORT_NAMES))
limitations_df.insert(2, "Score", limitations_df["Paper No."].map(df.set_index("paper_id")["Priority Score"]))
limitations_df.insert(3, "Score Meaning", limitations_df["Paper No."].map(df.set_index("paper_id")["Score Meaning"]))

display(limitations_df)

,Paper No.,Short Name,Score,Score Meaning,No_Economic_Outcomes,Simulated_Benchmark,Fixed_Candidate_Set,Observational_NonCausal,LLM_as_Judge,Narrow_Domain,No_Live_Platform,No_Content_Intervention,Platform_Instability,Limited_Multimodal,Limited_Finance_Relevance
0,1,GEO,10.0,Must-read / central paper,1.0,1.0,1.0,0,1.0,0,0.5,0,0.5,1.0,0.5
1,2,SAGEO Arena,9.5,Must-read / central paper,1.0,1.0,0.0,0,1.0,0,1.0,0,0.5,1.0,0.0
2,3,Citation Selection→Absorption,9.0,Must-read / central paper,1.0,0.0,0.0,1,0.5,0,0.0,1,0.5,1.0,0.0
3,4,FeatGEO,8.5,"Highly relevant, should include in memo",1.0,1.0,0.5,0,1.0,0,1.0,0,0.5,1.0,0.5
4,5,AgenticGEO,9.0,Must-read / central paper,1.0,1.0,0.5,0,1.0,0,1.0,0,0.5,1.0,0.5
5,6,GEO-SFE,8.0,"Highly relevant, should include in memo",1.0,1.0,0.5,0,1.0,0,1.0,0,0.5,1.0,0.5
6,7,EcoGEO,8.5,"Highly relevant, should include in memo",1.0,1.0,0.0,0,1.0,1,1.0,0,0.5,1.0,0.5
7,8,AutoGEO,10.0,Must-read / central paper,1.0,1.0,0.5,0,1.0,0,1.0,0,0.5,1.0,0.5
8,9,E-GEO,9.5,Must-read / central paper,0.5,1.0,0.5,0,1.0,1,1.0,0,0.5,1.0,0.0
9,10,AI Disrupts Search,9.5,Must-read / central paper,0.5,0.0,0.0,1,0.5,0,0.0,1,1.0,0.5,0.0


## 10. Finance/econ research gaps

This sheet translates the comparative analysis into possible finance/econ project directions.

In [15]:
finance_gaps = pd.DataFrame([
    {
        "Gap Area": "AI visibility → economic outcomes",
        "CS Literature Covers": "Visibility scores, citation rates, answer inclusion, rank changes, source overlap.",
        "Missing Econ Outcome": "Actual traffic, clicks, conversion, sales, advertising revenue, firm value, or consumer demand.",
        "Relevant Papers": "1, 8, 9, 10, 11",
        "Possible Empirical Design": "Construct a firm/product/source-level AI visibility panel across ChatGPT, Perplexity, Gemini, Google AIO, and Google SERP; link visibility to web traffic or other economic proxies.",
        "Candidate Data": "AI query panel; Google SERP/AIO scraping; Similarweb/traffic proxies; product rank/sales proxies; firm websites.",
        "Potential Dependent Variables": "AI citation rate; answer inclusion; product rank; source overlap; traffic change.",
        "Notes/Risks": "Platform instability; query sampling; terms-of-service constraints; hard-to-observe true conversion or revenue.",
    },
    {
        "Gap Area": "Firm website adaptation",
        "CS Literature Covers": "Content rewriting, structural features, preference rules, stage-aware optimization.",
        "Missing Econ Outcome": "Whether real firms actually change websites after AI search adoption and whether those changes pay off.",
        "Relevant Papers": "2, 4, 5, 6, 7, 8",
        "Possible Empirical Design": "Track changes in schema markup, FAQs, headings, product descriptions, investor-relations pages, and evidence density before/after AI search expansion.",
        "Candidate Data": "Common Crawl; Internet Archive; firm websites; structured data validators; AIO visibility panel.",
        "Potential Dependent Variables": "Adoption of schema/FAQ; AI citation probability; ranking/citation changes after content changes.",
        "Notes/Risks": "Causal identification is difficult; firms may change websites for many non-AI reasons.",
    },
    {
        "Gap Area": "E-commerce and product-market competition",
        "CS Literature Covers": "GEO for e-commerce and product ranking under generative recommendation.",
        "Missing Econ Outcome": "Seller competition, conversion, product sales, and consumer welfare effects.",
        "Relevant Papers": "8, 9, 10",
        "Possible Empirical Design": "Compare product visibility in AI shopping/recommendation answers with traditional search and marketplace rankings; test whether description structure predicts AI recommendation probability.",
        "Candidate Data": "Amazon/retail product pages; AI shopping prompts; product reviews; prices; ratings; rank proxies.",
        "Potential Dependent Variables": "AI recommendation probability; product rank; source inclusion; conversion/sales proxy.",
        "Notes/Risks": "Sales data may be unavailable; product queries need careful sampling.",
    },
    {
        "Gap Area": "Publisher impact and platform bargaining",
        "CS Literature Covers": "AIO activation, source quality, publisher impact, source concentration, news citation patterns.",
        "Missing Econ Outcome": "Publisher traffic loss, ad revenue displacement, licensing bargaining, and market concentration.",
        "Relevant Papers": "10, 11, 13, 14",
        "Possible Empirical Design": "Measure publisher citation frequency in AI answers and compare with traffic/referral trends or ad-supported page exposure.",
        "Candidate Data": "AIO citations; Google SERP; publisher metadata; traffic estimates; ad-tech detection.",
        "Potential Dependent Variables": "Publisher AI citation share; first-page SERP displacement; traffic/revenue proxy.",
        "Notes/Risks": "Traffic estimates are noisy; publisher contracts/licensing are often private.",
    },
    {
        "Gap Area": "Financial disclosure visibility",
        "CS Literature Covers": "Financial-document RAG, multimodal retrieval, long-context vs RAG, visual citation.",
        "Missing Econ Outcome": "Investor attention, analyst coverage, disclosure design choices, or market reaction linked to AI retrievability/citability.",
        "Relevant Papers": "18, 19, 20, 23",
        "Possible Empirical Design": "Evaluate whether firm disclosures are retrievable and citable by AI systems; link retrievability to disclosure structure, firm size, analyst coverage, or investor attention proxies.",
        "Candidate Data": "10-K/10-Q filings; investor decks; annual reports; FinRAG-style QA prompts; firm characteristics.",
        "Potential Dependent Variables": "Retrieval hit rate; visual citation precision; answer support rate; AI disclosure visibility score.",
        "Notes/Risks": "Requires careful document parsing and prompt design; multimodal evaluation can be expensive.",
    },
    {
        "Gap Area": "Temporal misinformation and stale financial facts",
        "CS Literature Covers": "Outdated information harms RAG; daily-changing facts need temporal retrieval and reranking.",
        "Missing Econ Outcome": "Impact of stale AI answers on consumer/investor beliefs or decision-making.",
        "Relevant Papers": "21, 22",
        "Possible Empirical Design": "Audit AI answers for time-sensitive firm, executive, product, regulatory, interest-rate, and market facts across repeated dates.",
        "Candidate Data": "Daily query panel; Wikipedia revisions; firm news; SEC filings; market data; AI answers.",
        "Potential Dependent Variables": "Outdated answer rate; stale-source citation rate; date-specific correctness.",
        "Notes/Risks": "Ground truth changes over time; needs timestamped data collection.",
    },
    {
        "Gap Area": "Cross-platform AI visibility",
        "CS Literature Covers": "Search platforms differ in source selection, citation behavior, and user preferences.",
        "Missing Econ Outcome": "Whether different AI platforms systematically favor different firms, publishers, or source types.",
        "Relevant Papers": "3, 10, 11, 12, 13, 14",
        "Possible Empirical Design": "Run matched prompts across ChatGPT, Perplexity, Gemini, Google AIO, and Google Search; compare source overlap, citation positions, and answer influence.",
        "Candidate Data": "Matched query panel; AI outputs; citation extraction; source metadata; platform identifiers.",
        "Potential Dependent Variables": "Cross-platform visibility variance; source overlap; platform-specific citation probability.",
        "Notes/Risks": "API/UI access changes; personalization and geography may affect results.",
    },
])

display(finance_gaps)

,Gap Area,CS Literature Covers,Missing Econ Outcome,Relevant Papers,Possible Empirical Design,Candidate Data,Potential Dependent Variables,Notes/Risks
0,AI visibility → economic outcomes,"Visibility scores, citation rates, answer inclusion, rank changes, source overlap.","Actual traffic, clicks, conversion, sales, advertising revenue, firm value, or consumer demand.","1, 8, 9, 10, 11","Construct a firm/product/source-level AI visibility panel across ChatGPT, Perplexity, Gemini, Google AIO, and Google...",AI query panel; Google SERP/AIO scraping; Similarweb/traffic proxies; product rank/sales proxies; firm websites.,AI citation rate; answer inclusion; product rank; source overlap; traffic change.,Platform instability; query sampling; terms-of-service constraints; hard-to-observe true conversion or revenue.
1,Firm website adaptation,"Content rewriting, structural features, preference rules, stage-aware optimization.",Whether real firms actually change websites after AI search adoption and whether those changes pay off.,"2, 4, 5, 6, 7, 8","Track changes in schema markup, FAQs, headings, product descriptions, investor-relations pages, and evidence density...",Common Crawl; Internet Archive; firm websites; structured data validators; AIO visibility panel.,Adoption of schema/FAQ; AI citation probability; ranking/citation changes after content changes.,Causal identification is difficult; firms may change websites for many non-AI reasons.
2,E-commerce and product-market competition,GEO for e-commerce and product ranking under generative recommendation.,"Seller competition, conversion, product sales, and consumer welfare effects.","8, 9, 10",Compare product visibility in AI shopping/recommendation answers with traditional search and marketplace rankings; t...,Amazon/retail product pages; AI shopping prompts; product reviews; prices; ratings; rank proxies.,AI recommendation probability; product rank; source inclusion; conversion/sales proxy.,Sales data may be unavailable; product queries need careful sampling.
3,Publisher impact and platform bargaining,"AIO activation, source quality, publisher impact, source concentration, news citation patterns.","Publisher traffic loss, ad revenue displacement, licensing bargaining, and market concentration.","10, 11, 13, 14",Measure publisher citation frequency in AI answers and compare with traffic/referral trends or ad-supported page exp...,AIO citations; Google SERP; publisher metadata; traffic estimates; ad-tech detection.,Publisher AI citation share; first-page SERP displacement; traffic/revenue proxy.,Traffic estimates are noisy; publisher contracts/licensing are often private.
4,Financial disclosure visibility,"Financial-document RAG, multimodal retrieval, long-context vs RAG, visual citation.","Investor attention, analyst coverage, disclosure design choices, or market reaction linked to AI retrievability/cita...","18, 19, 20, 23",Evaluate whether firm disclosures are retrievable and citable by AI systems; link retrievability to disclosure struc...,10-K/10-Q filings; investor decks; annual reports; FinRAG-style QA prompts; firm characteristics.,Retrieval hit rate; visual citation precision; answer support rate; AI disclosure visibility score.,Requires careful document parsing and prompt design; multimodal evaluation can be expensive.
5,Temporal misinformation and stale financial facts,Outdated information harms RAG; daily-changing facts need temporal retrieval and reranking.,Impact of stale AI answers on consumer/investor beliefs or decision-making.,"21, 22","Audit AI answers for time-sensitive firm, executive, product, regulatory, interest-rate, and market facts across rep...",Daily query panel; Wikipedia revisions; firm news; SEC filings; market data; AI answers.,Outdated answer rate; stale-source citation rate; date-specific correctness.,Ground truth changes over time; needs timestamped data collection.
6,Cross-platform AI visibility,"Search platforms differ in source selection, citation b

## 11. Priority view based only on numerical score

This view implements the user-provided score criteria and ignores the original `Priority Label`.

In [16]:
priority_view = df[[
    "paper_id", "Short Name", "Paper Title", "Priority Score", "Score Meaning",
    "Evidence Type", "Metric Family", "Finance Relevance Level",
    "Relevance for Finance/Econ Project", "Limitations / Open Gaps"
]].copy()
priority_view = priority_view.rename(columns={"paper_id": "Paper No.", "Priority Score": "Score"})
priority_view = priority_view.sort_values(["Score", "Paper No."], ascending=[False, True])

display(priority_view)

,Paper No.,Short Name,Paper Title,Score,Score Meaning,Evidence Type,Metric Family,Finance Relevance Level,Relevance for Finance/Econ Project,Limitations / Open Gaps
0,1,GEO,GEO: Generative Engine Optimization,10.0,Must-read / central paper,GEO optimization / website adaptation,AI visibility / impression,Medium,Highly relevant. This paper directly supports a finance/econ project studying how firms may adapt website content an...,"The paper tests only a limited number of generative engine settings, so methods may need to adapt as generative engi..."
7,8,AutoGEO,What Generative Search Engines Like and How to Optimize Web Content Cooperatively,10.0,Must-read / central paper,Preference-rule GEO optimization,GEO visibility + generative-engine utility,High,Very highly relevant. This paper directly studies how content providers can strategically adapt web content to gain ...,The experiments are benchmark-based and use constructed generative engines rather than long-term audits of live prod...
1,2,SAGEO Arena,SAGEO Arena: A Realistic Environment for Evaluating Search-Augmented Generative Engine Optimization,9.5,Must-read / central paper,Full-pipeline SAGEO / structured search,Stage-level search visibility,High,"Highly relevant. This paper is directly useful for a finance/econ project on how firms may adapt website content, me...","The paper uses a constructed benchmark rather than live commercial systems such as ChatGPT, Perplexity, Gemini, or G..."
8,9,E-GEO,E-GEO: A Testbed for Generative Engine Optimization in E-Commerce,9.5,Must-read / central paper,E-commerce GEO / product ranking,Product ranking / recommendation visibility,High,Very highly relevant. This is one of the strongest finance/econ-facing GEO papers because it directly studies e-comm...,The evaluation uses a simulated GPT-4o-based generative shopping engine rather than a live deployment such as Amazon...
9,10,AI Disrupts Search,"How Generative AI Disrupts Search: An Empirical Study of Google Search, Gemini, and AI Overviews",9.5,Must-read / central paper,Live AI search platform audit,Source overlap / AIO activation / robustness,High,Very highly relevant. This paper provides direct empirical evidence that AI search reallocates website visibility re...,"The paper measures retrieved/cited source visibility rather than actual user clicks, traffic, conversions, ad revenu..."
2,3,Citation Selection→Absorption,From Citation Selection to Citation Absorption: A Measurement Framework for Generative Engine Optimization Across AI...,9.0,Must-read / central paper,AI visibility measurement / citation absorption,Citation selection and absorption,High,Highly relevant. The paper provides measurable dependent variables for AI-mediated visibility beyond traditional SEO...,The paper is primarily descriptive and does not establish causal effects of page features on citation outcomes. The ...
4,5,AgenticGEO,AgenticGEO: A Self-Evolving Agentic System for Generative Engine Optimization,9.0,Must-read / central paper,Agentic GEO optimization,Adaptive GEO visibility gain,Medium,"Medium-high relevance. The paper is useful for understanding how firms, advertisers, or content providers might dyna...","The study is a controlled benchmark evaluation rather than a live audit of ChatGPT, Perplexity, Gemini, or Google AI..."
10,11,Google AIO Measurement,"Measuring Google AI Overviews: Activation, Source Quality, Claim Fidelity, and Publisher Impact",9.0,Must-read / central paper,Live AI search platform audit,AIO activation / source quality / claim fidelity,High,"Very highly relevant. This paper directly links AI-mediated search to publisher economics, source visibility, advert...","The study measures AIO behavior and publisher ad exposure but does not directly observe user clicks, website traffic..."
19,20,FinRAGBench-V,FinRAGBench-V: A Benchmark for Multimodal RAG with Visual Citation in the Financial Domain,9.0,Must-read / central paper,Finance multimodal RAG benchmark,Financial RAG retrieval + visual c

## 12. Analysis overview table

This dashboard-style sheet summarizes the main comparative conclusions.

In [17]:
analysis_overview = pd.DataFrame([
    {
        "Theme": "Main conclusion",
        "Finding": "The CS literature is methodologically rich but economically incomplete.",
        "Evidence": "Most papers measure visibility, citation, retrieval, rank, or answer quality rather than clicks, sales, revenue, or firm outcomes.",
        "Relevant Papers": "1–23",
    },
    {
        "Theme": "Visibility is multi-stage",
        "Finding": "AI visibility can fail at retrieval, reranking, citation selection, answer absorption, or citation fidelity.",
        "Evidence": "SAGEO, citation absorption, ALCE, AIO measurement, and FinRAGBench-V separate different visibility/evidence stages.",
        "Relevant Papers": "2, 3, 11, 20, 23",
    },
    {
        "Theme": "Strongest finance/econ bridges",
        "Finding": "E-commerce and financial-document RAG provide the clearest bridge from CS benchmarks to economic questions.",
        "Evidence": "E-GEO measures product ranking; FinRAGBench-V/REAL-MM-RAG/LaRA evaluate financial documents and visual citation.",
        "Relevant Papers": "9, 18, 19, 20",
    },
    {
        "Theme": "External validity trade-off",
        "Finding": "Controlled GEO benchmarks have stronger experimental control but weaker real-world validity; live platform audits have stronger real-world relevance but weaker causal identification.",
        "Evidence": "GEO/AutoGEO/E-GEO vs. Google AIO/Gemini/Search Arena audits.",
        "Relevant Papers": "1, 8, 9, 10, 11, 12, 13, 14",
    },
    {
        "Theme": "Recommended next step",
        "Finding": "Build a firm/product/source-level AI visibility panel and link technical visibility metrics to economic outcomes or proxies.",
        "Evidence": "Metric crosswalk and finance/econ gap table in this workbook.",
        "Relevant Papers": "1, 2, 8, 9, 10, 11, 20, 23",
    },
])

display(analysis_overview)

,Theme,Finding,Evidence,Relevant Papers
0,Main conclusion,The CS literature is methodologically rich but economically incomplete.,"Most papers measure visibility, citation, retrieval, rank, or answer quality rather than clicks, sales, revenue, or ...",1–23
1,Visibility is multi-stage,"AI visibility can fail at retrieval, reranking, citation selection, answer absorption, or citation fidelity.","SAGEO, citation absorption, ALCE, AIO measurement, and FinRAGBench-V separate different visibility/evidence stages.","2, 3, 11, 20, 23"
2,Strongest finance/econ bridges,E-commerce and financial-document RAG provide the clearest bridge from CS benchmarks to economic questions.,E-GEO measures product ranking; FinRAGBench-V/REAL-MM-RAG/LaRA evaluate financial documents and visual citation.,"9, 18, 19, 20"
3,External validity trade-off,Controlled GEO benchmarks have stronger experimental control but weaker real-world validity; live platform audits ha...,GEO/AutoGEO/E-GEO vs. Google AIO/Gemini/Search Arena audits.,"1, 8, 9, 10, 11, 12, 13, 14"
4,Recommended next step,Build a firm/product/source-level AI visibility panel and link technical visibility metrics to economic outcomes or ...,Metric crosswalk and finance/econ gap table in this workbook.,"1, 2, 8, 9, 10, 11, 20, 23"


## 13. Export to Excel workbook

The workbook includes:
- `Analysis_Overview`
- `Score_Criteria`
- `Paper_Taxonomy`
- `Benchmark_Comparison`
- `Metric_Crosswalk`
- `Similarity_Features`
- `Similarity_Matrix`
- `Limitations_Heatmap`
- `Finance_Econ_Gaps`
- `Priority_View`

Run this cell to create the polished `.xlsx`.

In [18]:
OUTPUT_XLSX = Path("JunseoPark_GEO_AI_Search_Comparative_Analysis.xlsx")

score_criteria = pd.DataFrame([
    {"Score Range": "9–10", "Meaning": "Must-read / central paper"},
    {"Score Range": "8–8.9", "Meaning": "Highly relevant, should include in memo"},
    {"Score Range": "7–7.9", "Meaning": "Useful supporting paper"},
    {"Score Range": "6–6.9", "Meaning": "Background only"},
    {"Score Range": "<6", "Meaning": "Probably mention briefly or exclude"},
])

sheets = {
    "Analysis_Overview": analysis_overview,
    "Score_Criteria": score_criteria,
    "Paper_Taxonomy": taxonomy,
    "Benchmark_Comparison": benchmark_comparison,
    "Metric_Crosswalk": metric_crosswalk,
    "Similarity_Features": features_df,
    "Similarity_Matrix": similarity_df.round(3).reset_index().rename(columns={"index": "Paper"}),
    "Limitations_Heatmap": limitations_df,
    "Finance_Econ_Gaps": finance_gaps,
    "Priority_View": priority_view,
}

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    for sheet_name, data in sheets.items():
        data.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Wrote raw workbook: {OUTPUT_XLSX.resolve()}")

Wrote raw workbook: C:\Users\junse\Documents\research\AI_Search_GEO_Literature_Review\09_code\JunseoPark_GEO_AI_Search_Comparative_Analysis.xlsx


## 14. Style the Excel workbook

This is separate from the export step so that content generation and styling remain easy to debug.

In [19]:
def style_workbook(path):
    wb = load_workbook(path)

    navy = "10233F"
    blue = "255A9B"
    mid_gray = "D8DEE8"
    white = "FFFFFF"

    header_fill = PatternFill("solid", fgColor=navy)
    header_font = Font(color=white, bold=True)
    thin_border = Border(
        left=Side(style="thin", color=mid_gray),
        right=Side(style="thin", color=mid_gray),
        top=Side(style="thin", color=mid_gray),
        bottom=Side(style="thin", color=mid_gray),
    )

    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False

        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
            cell.border = thin_border

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical="top", wrap_text=True)
                cell.border = thin_border
                if cell.row % 2 == 0:
                    cell.fill = PatternFill("solid", fgColor="FBFCFE")

        if ws.max_row >= 1 and ws.max_column >= 1:
            ws.auto_filter.ref = ws.dimensions

        for col_idx in range(1, ws.max_column + 1):
            col_letter = get_column_letter(col_idx)
            header_value = str(ws.cell(row=1, column=col_idx).value or "")
            max_len = len(header_value)

            sample_limit = min(ws.max_row, 60)
            for row_idx in range(2, sample_limit + 1):
                value = ws.cell(row=row_idx, column=col_idx).value
                if value is not None:
                    max_len = max(max_len, min(len(str(value)), 80))

            if any(key in header_value.lower() for key in ["title", "finding", "evidence", "reason", "interpretation", "design", "notes", "limitation", "relevance", "composition"]):
                width = min(max(max_len * 0.75, 24), 55)
            elif "link" in header_value.lower() or "url" in header_value.lower():
                width = min(max(max_len * 0.7, 18), 38)
            elif "paper" in header_value.lower() or "short" in header_value.lower():
                width = min(max(max_len * 0.8, 12), 28)
            else:
                width = min(max(max_len * 0.9, 10), 24)

            ws.column_dimensions[col_letter].width = width

        for row_idx in range(2, ws.max_row + 1):
            ws.row_dimensions[row_idx].height = 34
        ws.row_dimensions[1].height = 28

    if "Similarity_Matrix" in wb.sheetnames:
        ws = wb["Similarity_Matrix"]
        if ws.max_row > 2 and ws.max_column > 2:
            data_range = f"B2:{get_column_letter(ws.max_column)}{ws.max_row}"
            ws.conditional_formatting.add(
                data_range,
                ColorScaleRule(
                    start_type="num", start_value=0, start_color="FFFFFF",
                    mid_type="num", mid_value=0.5, mid_color="DCEBFF",
                    end_type="num", end_value=1, end_color=blue,
                )
            )
            for row in ws.iter_rows(min_row=2, min_col=2):
                for cell in row:
                    cell.number_format = "0.00"
                    cell.alignment = Alignment(horizontal="center", vertical="center")

    if "Limitations_Heatmap" in wb.sheetnames:
        ws = wb["Limitations_Heatmap"]
        if ws.max_row > 2 and ws.max_column >= 5:
            data_range = f"E2:{get_column_letter(ws.max_column)}{ws.max_row}"
            ws.conditional_formatting.add(
                data_range,
                ColorScaleRule(
                    start_type="num", start_value=0, start_color="FFFFFF",
                    mid_type="num", mid_value=0.5, mid_color="FFF4CC",
                    end_type="num", end_value=1, end_color="F4B183",
                )
            )
            for row in ws.iter_rows(min_row=2, min_col=5):
                for cell in row:
                    cell.number_format = "0.0"
                    cell.alignment = Alignment(horizontal="center", vertical="center")

    if "Similarity_Features" in wb.sheetnames:
        ws = wb["Similarity_Features"]
        if ws.max_row > 2 and ws.max_column >= 5:
            data_range = f"E2:{get_column_letter(ws.max_column)}{ws.max_row}"
            ws.conditional_formatting.add(
                data_range,
                ColorScaleRule(
                    start_type="num", start_value=0, start_color="FFFFFF",
                    mid_type="num", mid_value=0.5, mid_color="E6EDF6",
                    end_type="num", end_value=1, end_color="9DC3E6",
                )
            )
            for row in ws.iter_rows(min_row=2, min_col=5):
                for cell in row:
                    cell.number_format = "0.0"
                    cell.alignment = Alignment(horizontal="center", vertical="center")

    ws = wb["Analysis_Overview"]
    ws.insert_rows(1, 2)
    ws["A1"] = "Comparative Analysis of GEO, AI Search Visibility, and RAG Benchmarks"
    ws["A1"].font = Font(bold=True, size=15, color=navy)
    ws["A2"] = "Scoring uses the numerical Priority Score only; the original Priority Label column is ignored."
    ws["A2"].font = Font(italic=True, color="647084")
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=min(4, ws.max_column))
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=min(4, ws.max_column))

    wb.save(path)

style_workbook(OUTPUT_XLSX)
print(f"Styled workbook saved: {OUTPUT_XLSX.resolve()}")

Styled workbook saved: C:\Users\junse\Documents\research\AI_Search_GEO_Literature_Review\09_code\JunseoPark_GEO_AI_Search_Comparative_Analysis.xlsx


## 15. Quick verification

Run this cell after export to confirm sheet names and output path.

In [20]:
if OUTPUT_XLSX.exists():
    wb_check = load_workbook(OUTPUT_XLSX, read_only=True)
    print("Created workbook:", OUTPUT_XLSX.resolve())
    print("Sheets:")
    for s in wb_check.sheetnames:
        print(" -", s)
else:
    print("Workbook not found. Run the export cells above.")

Created workbook: C:\Users\junse\Documents\research\AI_Search_GEO_Literature_Review\09_code\JunseoPark_GEO_AI_Search_Comparative_Analysis.xlsx
Sheets:
 - Analysis_Overview
 - Score_Criteria
 - Paper_Taxonomy
 - Benchmark_Comparison
 - Metric_Crosswalk
 - Similarity_Features
 - Similarity_Matrix
 - Limitations_Heatmap
 - Finance_Econ_Gaps
 - Priority_View
